# Evaluate StarCE PredMethod

Testing the impact of StarCE predicate handling methods on estimation accuracy.

StarCE has two predicate handling approaches (PM2 experimentally verified as equivalent to PM0, not tested separately):

- **PredMethod=0**: Builds `singleStats` for tables with predicates (using filtered cardinality), shrinks via `AdjustToAverage(PredicateAdjustRate)` then merges into join statistics. `k=0` -> flattens degree sequences to mean values; `k=1` -> fully preserves original skew.
- **PredMethod=1** (current default): Computes `filter_coeff = filtered_card / NDV`, directly multiplied into DSStatistic.

`EXTRA_PM0_PAR_RATES` can configure additional PM0 variants to compare PM0 performance under different `PredicateAdjustRate` values.

**Note**: PredMethod only affects the estimation phase, not statistics collection; `RefreshStatistics=1` is not needed.

In [1]:
import os
import sys
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.colors import to_hex

sys.path.append(os.path.abspath('.'))

from ExperimentRunner import StarCETestRunner, setup_starce_executable

In [2]:
# ========== Experiment Configuration ==========

# Whether to actually run StarCE
RUN_STARCE = True

# Benchmarks to test
BENCHMARKS = ["STATS", "JOBM", "JOBLight", "JOBLightRanges"]

# AdjustRate: global default Merge shrink rate (for variants that do not specify AdjustRate)
ADJUST_RATE = 1
ADJUST_RATE_2 = 0.1
ADJUST_RATE_3 = 0.0

# Additional PM0 variants under different AdjustRate (replacing original PM2 variant positions)
# PredicateAdjustRate fixed at 1.0, only AdjustRate varies
# PM0_ADJUST_RATES = [0.0, 0.2, 0.5, 0.8, 1.0]

# Variant format: (label, PredMethod, PredicateAdjustRate, AdjustRate)
# None -> use ADJUST_RATE as default value
EXPERIMENT_VARIANTS = [
    ("PM1",        1, None, None),
    ("PAR0.0", 0, 0.0,  None),
    ("PAR0.1", 0, 0.1,  None),
    ("PAR0.5", 0, 0.5,  None),
    ("PAR1.0", 0, 1.0,  None),
    (f"AR{ADJUST_RATE_2:.1f}-PAR0.0", 0, 0.0, ADJUST_RATE_2),
    (f"AR{ADJUST_RATE_2:.1f}-PAR0.1", 0, 0.1, ADJUST_RATE_2),
    (f"AR{ADJUST_RATE_2:.1f}-PAR0.5", 0, 0.5, ADJUST_RATE_2),
    (f"AR{ADJUST_RATE_2:.1f}-PAR1.0", 0, 1.0, ADJUST_RATE_2),
    (f"AR{ADJUST_RATE_3:.1f}-PAR0.0", 0, 0.0, ADJUST_RATE_3),
    (f"AR{ADJUST_RATE_3:.1f}-PAR0.1", 0, 0.1, ADJUST_RATE_3),
    (f"AR{ADJUST_RATE_3:.1f}-PAR0.5", 0, 0.5, ADJUST_RATE_3),
    (f"AR{ADJUST_RATE_3:.1f}-PAR1.0", 0, 1.0, ADJUST_RATE_3),
# ] + [
#     (f"PM0-AR{ar:.1f}", 0, 1.0, ar) for ar in PM0_ADJUST_RATES
]

In [3]:
def get_project_root() -> Path:
    cwd = Path.cwd()
    for parent in [cwd] + list(cwd.parents):
        if (parent / "Benchmark").exists() and (parent / "experiment").exists():
            return parent
    if "experiment" in str(cwd):
        return cwd.parent
    if (cwd / "Benchmark").exists() or (cwd / "experiment").exists():
        return cwd
    return cwd

PROJECT_ROOT = get_project_root()
print(f"Project root directory: {PROJECT_ROOT}")

CHECKPOINT_ROOT = PROJECT_ROOT / "experiment" / "checkpoint" / "StarCE" / "pred_method"
FIGURE_DIR = PROJECT_ROOT / "experiment" / "checkpoint" / "figures"
CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

print(f"Checkpoint root: {CHECKPOINT_ROOT}")
print(f"Figure dir: {FIGURE_DIR}")
print(f"Benchmarks: {BENCHMARKS}")
print(f"ADJUST_RATE: {ADJUST_RATE}")
print(f"Variants:")
for label, pm, par, ar in EXPERIMENT_VARIANTS:
    par_display = ADJUST_RATE if par is None else par
    ar_display = ADJUST_RATE if ar is None else ar
    print(f"  {label}: PredMethod={pm}, PredicateAdjustRate={par_display}, AdjustRate={ar_display}")

Project root directory: /home/user/project/starCE
Checkpoint root: /home/user/project/starCE/experiment/checkpoint/StarCE/pred_method
Figure dir: /home/user/project/starCE/experiment/checkpoint/figures
Benchmarks: ['STATS', 'JOBM', 'JOBLight', 'JOBLightRanges']
ADJUST_RATE: 1
Variants:
  PM1: PredMethod=1, PredicateAdjustRate=1, AdjustRate=1
  PAR0.0: PredMethod=0, PredicateAdjustRate=0.0, AdjustRate=1
  PAR0.1: PredMethod=0, PredicateAdjustRate=0.1, AdjustRate=1
  PAR0.5: PredMethod=0, PredicateAdjustRate=0.5, AdjustRate=1
  PAR1.0: PredMethod=0, PredicateAdjustRate=1.0, AdjustRate=1
  AR0.1-PAR0.0: PredMethod=0, PredicateAdjustRate=0.0, AdjustRate=0.1
  AR0.1-PAR0.1: PredMethod=0, PredicateAdjustRate=0.1, AdjustRate=0.1
  AR0.1-PAR0.5: PredMethod=0, PredicateAdjustRate=0.5, AdjustRate=0.1
  AR0.1-PAR1.0: PredMethod=0, PredicateAdjustRate=1.0, AdjustRate=0.1
  AR0.0-PAR0.0: PredMethod=0, PredicateAdjustRate=0.0, AdjustRate=0.0
  AR0.0-PAR0.1: PredMethod=0, PredicateAdjustRate=0.1, Adj

In [4]:
if RUN_STARCE:
    runner = StarCETestRunner(str(PROJECT_ROOT))
    setup_starce_executable(PROJECT_ROOT, runner.running_space)
else:
    runner = StarCETestRunner(str(PROJECT_ROOT))

Successfully copied starce executable to: /home/user/project/starCE/experiment/running_space/starce


In [5]:
def resolve_real_path(benchmark: str) -> Path:
    mapping = {
        "STATS":          PROJECT_ROOT / "Benchmark" / "workloads" / "STATS-CEB" / "subquery" / "result" / "real.txt",
        "JOBM":           PROJECT_ROOT / "Benchmark" / "workloads" / "JOBM" / "subquery" / "result" / "real.txt",
        "JOBLight":       PROJECT_ROOT / "Benchmark" / "workloads" / "JOBLight" / "subquery" / "result" / "real.txt",
        "JOBLightRanges": PROJECT_ROOT / "Benchmark" / "workloads" / "JOBLightRanges" / "subquery" / "result" / "real.txt",
    }
    if benchmark not in mapping:
        raise ValueError(f"Unknown benchmark: {benchmark}")
    return mapping[benchmark]


def get_benchmark_config(benchmark: str):
    if benchmark == "STATS":
        return runner.get_stats_config()
    if benchmark == "JOBM":
        return runner.get_jobm_config()
    if benchmark == "JOBLight":
        return runner.get_joblight_config()
    if benchmark == "JOBLightRanges":
        return runner.get_joblight_ranges_config()
    raise ValueError(f"Unknown benchmark: {benchmark}")


def output_path(benchmark: str, label: str) -> Path:
    out_dir = CHECKPOINT_ROOT / benchmark
    out_dir.mkdir(parents=True, exist_ok=True)
    return out_dir / f"card_{benchmark}_{label}.txt"


def read_txt_file(file_path: Path) -> list:
    values = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            stripped = line.strip()
            if stripped:
                values.append(float(stripped))
    return values


def generate_color_palette(n: int) -> list:
    if n <= 10:
        cmap = mpl.colormaps['tab10']
        return [to_hex(cmap(i)) for i in range(n)]
    cmap = mpl.colormaps['tab20']
    return [to_hex(cmap(i % 20)) for i in range(n)]

In [6]:
run_results = []

for benchmark in BENCHMARKS:
    for label, pred_method, pred_adjust_rate, adjust_rate in EXPERIMENT_VARIANTS:
        out = output_path(benchmark, label)

        if RUN_STARCE:
            print(f"\n=== Run StarCE: benchmark={benchmark}, variant={label} ===")

            config = get_benchmark_config(benchmark)

            # Predicate method parameters
            config.PredMethod = pred_method
            config.PREDICATE_ADJUST_RATE = ADJUST_RATE if pred_adjust_rate is None else pred_adjust_rate
            config.ADJUST_RATE = ADJUST_RATE if adjust_rate is None else adjust_rate

            # No need to recollect statistics
            config.RefreshStatistics = 0

            # Run mode: RecordingSubquery
            explain_sql = runner.running_space / "explain.sql"
            runner._prepare_input_sql_with_explain(Path(config.SQL_PATH), explain_sql)

            config.RecordingSubquery = 1
            config.SUBQUERY_PATH = "null.sql"
            config.SQL_PATH = "explain.sql"
            config.SUBQUERY_RESULT_PATH = str(out)

            runner.save_config(config)

            start = time.time()
            success, _elapsed = runner.run_starce(suppress_output=True)
            total = time.time() - start

            if not success:
                raise RuntimeError(f"StarCE run failed: benchmark={benchmark}, variant={label}")

            print(f"Output: {out}  ({total:.3f}s)")
            run_results.append({"benchmark": benchmark, "variant": label, "time_sec": total})
        else:
            print(f"Skip (RUN_STARCE=False): benchmark={benchmark}, variant={label}")
            print(f"  Expected: {out}")

print(f"\nAll runs complete. Total: {len(BENCHMARKS)} benchmarks × {len(EXPERIMENT_VARIANTS)} variants")


=== Run StarCE: benchmark=STATS, variant=PM1 ===
[2026-04-25 14:46:57] Copied queries file to /home/user/project/starCE/experiment/running_space/explain.sql and added EXPLAIN
[2026-04-25 14:46:57] Config file saved to: /home/user/project/starCE/experiment/running_space/config.json
[2026-04-25 14:46:57] starce Run succeeded，elapsed: 0.20 sec
Output: /home/user/project/starCE/experiment/checkpoint/StarCE/pred_method/STATS/card_STATS_PM1.txt  (0.202s)

=== Run StarCE: benchmark=STATS, variant=PAR0.0 ===
[2026-04-25 14:46:57] Copied queries file to /home/user/project/starCE/experiment/running_space/explain.sql and added EXPLAIN
[2026-04-25 14:46:57] Config file saved to: /home/user/project/starCE/experiment/running_space/config.json
[2026-04-25 14:46:57] starce Run succeeded，elapsed: 0.22 sec
Output: /home/user/project/starCE/experiment/checkpoint/StarCE/pred_method/STATS/card_STATS_PAR0.0.txt  (0.219s)

=== Run StarCE: benchmark=STATS, variant=PAR0.1 ===
[2026-04-25 14:46:57] Copied quer

In [7]:
# all_error_data: {benchmark: {label: [relative_errors]}}
# Read entirely from checkpoint, independent of run cell memory state
all_error_data = {}

for benchmark in BENCHMARKS:
    real_path = resolve_real_path(benchmark)
    if not real_path.exists():
        raise FileNotFoundError(f"Real card file not found: {real_path}")
    true_values = read_txt_file(real_path)

    per_benchmark = {}
    for label, *_ in EXPERIMENT_VARIANTS:
        est_path = output_path(benchmark, label)
        if not est_path.exists():
            raise FileNotFoundError(f"Checkpoint not found: {est_path}\nPlease run the run cell first（RUN_STARCE=True）")
        est_values = read_txt_file(est_path)

        if len(est_values) != len(true_values):
            raise ValueError(
                f"Row count mismatch: benchmark={benchmark}, variant={label}, "
                f"true={len(true_values)}, est={len(est_values)}"
            )

        rel_errors = [max(1.0, e) / max(1.0, t) for t, e in zip(true_values, est_values)]
        per_benchmark[label] = rel_errors

        log_vals = np.log10(np.maximum(np.array(rel_errors), 1e-10))
        print(
            f"{benchmark} {label}: n={len(rel_errors)}, "
            f"median={np.median(log_vals):.3f}, "
            f"p75={np.percentile(log_vals, 75):.3f}, "
            f"max={np.max(log_vals):.3f}"
        )

    all_error_data[benchmark] = per_benchmark

print(f"\nTotal benchmarks processed: {len(all_error_data)}")

STATS PM1: n=2471, median=0.290, p75=1.004, max=8.507
STATS PAR0.0: n=2471, median=0.232, p75=0.965, max=8.507
STATS PAR0.1: n=2471, median=0.444, p75=1.064, max=8.773
STATS PAR0.5: n=2471, median=0.668, p75=1.396, max=9.351
STATS PAR1.0: n=2471, median=0.562, p75=1.482, max=9.761
STATS AR0.1-PAR0.0: n=2471, median=0.085, p75=0.734, max=8.507
STATS AR0.1-PAR0.1: n=2471, median=0.217, p75=0.871, max=8.773
STATS AR0.1-PAR0.5: n=2471, median=0.445, p75=1.049, max=9.351
STATS AR0.1-PAR1.0: n=2471, median=0.351, p75=1.125, max=9.761
STATS AR0.0-PAR0.0: n=2471, median=0.080, p75=0.715, max=8.507
STATS AR0.0-PAR0.1: n=2471, median=0.202, p75=0.853, max=8.773
STATS AR0.0-PAR0.5: n=2471, median=0.420, p75=1.016, max=9.351
STATS AR0.0-PAR1.0: n=2471, median=0.334, p75=1.085, max=9.761
JOBM PM1: n=6424, median=0.300, p75=0.922, max=4.826
JOBM PAR0.0: n=6424, median=0.305, p75=0.928, max=4.827
JOBM PAR0.1: n=6424, median=1.343, p75=2.472, max=6.739
JOBM PAR0.5: n=6424, median=2.214, p75=3.435, max

In [ ]:
import plot_style

variant_labels = [label for label, *_ in EXPERIMENT_VARIANTS]
variant_colors = {label: color for label, color in zip(variant_labels, plot_style.generate_color_palette(len(variant_labels)))}

fig, ax = plt.subplots(layout="constrained", figsize=(max(15, len(all_error_data) * 2.5), 5))

positions = []
violin_data = []
violin_colors = []
benchmark_centers = []
benchmark_labels_plot = []

group_width = 0.9
group_gap = 0.6

for idx, benchmark in enumerate(sorted(all_error_data.keys()), start=1):
    base_pos = (idx - 1) * (group_width + group_gap) + 1
    offsets = np.linspace(-group_width / 2, group_width / 2, len(variant_labels))

    for offset, label in zip(offsets, variant_labels):
        pos = base_pos + float(offset)
        errors = np.array(all_error_data[benchmark][label])
        log_errors = np.log10(np.maximum(errors, 1e-10))
        positions.append(pos)
        violin_data.append(log_errors)
        violin_colors.append(variant_colors[label])

    benchmark_centers.append(base_pos)
    benchmark_labels_plot.append(benchmark)

plot_style.draw_violins(ax, violin_data, positions, violin_colors,
                        widths=group_width / len(variant_labels))

all_log = np.concatenate(violin_data)
ax.set_ylim(np.min(all_log) - 0.5, np.max(all_log) + 0.5)
ax.set_ylabel("Relative Error (log10)")
ax.axhline(y=0, color="black", linestyle="--", linewidth=0.5)
ax.set_xticks(benchmark_centers)
ax.set_xticklabels(benchmark_labels_plot)
ax.set_title(f"StarCE PredMethod Comparison (AdjustRate={ADJUST_RATE})")

legend_handles = [
    mpl.patches.Patch(color=variant_colors[label], label=label)
    for label in variant_labels
]
ax.legend(handles=legend_handles, fontsize=9, loc="best")

output_file = FIGURE_DIR / "pred_method_boxplot_benchmarks_grouped.pdf"
plt.savefig(output_file)
print(f"Saved: {output_file}")
plt.show()


In [9]:
print("=" * 60)
print("Statistical Summary (log10 Relative Error)")
print(f"AdjustRate={ADJUST_RATE}")
print("=" * 60)

for benchmark in BENCHMARKS:
    if benchmark not in all_error_data:
        continue
    print(f"\n### {benchmark} ###")
    for label, *_ in EXPERIMENT_VARIANTS:
        errors = np.log10(np.maximum(np.array(all_error_data[benchmark][label]), 1e-10))
        print(f"  {label}:")
        print(f"    Median={np.median(errors):.3f}  Mean={np.mean(errors):.3f}  "
              f"p25={np.percentile(errors, 25):.3f}  p75={np.percentile(errors, 75):.3f}  "
              f"Max={np.max(errors):.3f}")

Statistical Summary (log10 Relative Error)
AdjustRate=1

### STATS ###
  PM1:
    Median=0.290  Mean=0.575  p25=0.004  p75=1.004  Max=8.507
  PAR0.0:
    Median=0.232  Mean=0.531  p25=0.000  p75=0.965  Max=8.507
  PAR0.1:
    Median=0.444  Mean=0.701  p25=0.020  p75=1.064  Max=8.773
  PAR0.5:
    Median=0.668  Mean=0.944  p25=0.100  p75=1.396  Max=9.351
  PAR1.0:
    Median=0.562  Mean=0.986  p25=0.092  p75=1.482  Max=9.761
  AR0.1-PAR0.0:
    Median=0.085  Mean=0.373  p25=-0.024  p75=0.734  Max=8.507
  AR0.1-PAR0.1:
    Median=0.217  Mean=0.531  p25=0.001  p75=0.871  Max=8.773
  AR0.1-PAR0.5:
    Median=0.445  Mean=0.757  p25=0.034  p75=1.049  Max=9.351
  AR0.1-PAR1.0:
    Median=0.351  Mean=0.796  p25=0.046  p75=1.125  Max=9.761
  AR0.0-PAR0.0:
    Median=0.080  Mean=0.346  p25=-0.054  p75=0.715  Max=8.507
  AR0.0-PAR0.1:
    Median=0.202  Mean=0.501  p25=0.000  p75=0.853  Max=8.773
  AR0.0-PAR0.5:
    Median=0.420  Mean=0.729  p25=0.023  p75=1.016  Max=9.351
  AR0.0-PAR1.0:
    Medi